# 03 Generate Synthetic Data

Notebook-native synthetic image generation. This notebook replaces `scripts/generate_synth.py`.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import torch
from torchvision.utils import save_image
from PIL import Image
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from models.gan import Generator

ROOT


In [ ]:
CLASS_NAMES = ["apple", "banana", "orange"]


def generate_synth_pool_notebook(ckpt, n_per_class, out_dir, batch_size=64, seed=42, class_names=None):
    ckpt = Path(ckpt)
    out_root = Path(out_dir)
    class_names = class_names or CLASS_NAMES
    cfg = Config()
    device = torch.device(cfg.resolve_device())

    G = Generator(z_dim=cfg.z_dim, num_classes=len(class_names)).to(device)
    ckpt_state = torch.load(ckpt, map_location=device, weights_only=True)
    G.load_state_dict(ckpt_state["G"])
    G.eval()

    torch.manual_seed(seed)
    start_time = time.time()
    counts = {}

    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = out_root / cls_name
        cls_dir.mkdir(parents=True, exist_ok=True)

        generated = 0
        while generated < n_per_class:
            bs = min(batch_size, n_per_class - generated)
            z = torch.randn(bs, cfg.z_dim, device=device)
            y = torch.full((bs,), cls_idx, dtype=torch.long, device=device)
            with torch.no_grad():
                imgs = G(z, y)

            for i in range(bs):
                save_image(
                    imgs[i],
                    cls_dir / f"{cls_name}_synth_{generated + i:05d}.png",
                    normalize=True,
                    value_range=(-1, 1),
                )
            generated += bs
        counts[cls_name] = generated

    summary = {
        "checkpoint": str(ckpt),
        "out_dir": str(out_root),
        "n_per_class": n_per_class,
        "counts": counts,
        "seed": seed,
        "generate_time_sec": round(time.time() - start_time, 1),
    }
    with open(out_root / "generation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


In [ ]:
# Edit these values before running.
CKPT = ROOT / "runs" / "gan" / "checkpoints" / "best_fid.pt"
OUT_DIR = ROOT / "data_synth"
N_PER_CLASS = 1300
BATCH_SIZE = 64
SEED = 42

CKPT, OUT_DIR


In [ ]:
generation_summary = generate_synth_pool_notebook(
    ckpt=CKPT,
    n_per_class=N_PER_CLASS,
    out_dir=OUT_DIR,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
generation_summary


In [ ]:
for class_name in CLASS_NAMES:
    sample_path = sorted((OUT_DIR / class_name).glob("*.png"))[0]
    print(class_name, sample_path.name)
    display(Image.open(sample_path))
